# B3N — Negative-point prompting (test B)

## Why
Dense-scene error is mostly **merging** (one SAM2 mask over many objects), and no SAM2 variant tried so far changes that (multimask selection, re-prompting seeds, masked input, SAM2.1-Large). All guided masks so far used a **single positive point**. SAM2 also accepts **negative points** ("not this"): a positive point on one object with negatives on its neighbours should push SAM2 towards one object instead of the group.

## Pre-declared design (before any B3N result)
| Choice | Rule |
|---|---|
| Positive point | each cached `dense_recovery` prompt |
| Negative points | its nearest other prompts, **up to 4**, within **2 × the image's median nearest-neighbour prompt distance** (isolated prompts: no negatives; images with < 2 prompts: positive only) |
| Model / input | SAM2.1-Small (frozen), full image, multimask off — only the prompt changes |
| B0 masks | unchanged (frozen cache) |
| Counting | unchanged: GPV_R4 (NMS 0.5 + B0 pool), then H1b |
| Targets | T1, T2′, T3′ as before |

**Main risk:** two prompts on the same object (common in sparse scenes) — a negative on one's own object may cut it, adding fragments in S1–S2.

## Run order
T4 → 1.0–1.2 → 2.1–2.2 → 3.1 (~15–20 min, resumable) → 4.1.

# 1. Setup

In [1]:
# 1.0 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# ============================================================
# Step 1.1 — Paths, repo modules, train pool, frozen-system counts
# ============================================================

import importlib, json, subprocess, sys, time
from datetime import datetime, timezone
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project_ZeroShot_Counting")
RESULTS_ROOT = PROJECT_ROOT / "results"
FSC147_RAW = PROJECT_ROOT / "datasets" / "FSC147" / "raw"
FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
TD_ROOT = PROJECT_ROOT / "cache" / "FSC147_train_dev400"
B3S_DIR = RESULTS_ROOT / "B3S_Semantic_Improvement"
OUT_DIR = RESULTS_ROOT / "B3N_Negative_Point_Prompts"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git"
REPO_DIR = Path("/content/Surrey-MScProject-Zero-Shot-Object-Counting")
!rm -rf "{REPO_DIR}"
!git clone -q "{REPO_URL}" "{REPO_DIR}"
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
REPO_COMMIT = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()

import src.pipeline.io_cache as io_cache
import src.pipeline.sg as sg
import src.pipeline.gpv as gpv
import src.pipeline.hybrid as hybrid
import src.pipeline.b0 as b0
for m in (io_cache, sg, gpv, hybrid, b0):
    importlib.reload(m)

STRATA = ["S1_0_20", "S2_20_40", "S3_40_60", "S4_60_80", "S5_80_95", "S6_95_100"]
W = {"S1_0_20": .20, "S2_20_40": .20, "S3_40_60": .20, "S4_60_80": .20, "S5_80_95": .15, "S6_95_100": .05}
TD_DIRS = {"b0_masks": TD_ROOT / "b0_masks", "a3c_masks": TD_ROOT / "a3c_masks",
           "depth": TD_ROOT / "depth", "m1_split_labels": TD_ROOT / "m1_split_labels"}
SG_CFG = sg.SGConfig.from_frozen_json(RESULTS_ROOT / "B3R_SG_Rebuild_and_Fixes" / "SG_v1.1_frozen_config.json")

td_df = pd.read_csv(TD_ROOT / "FSC147_train_dev400.csv")
td_res = pd.read_csv(B3S_DIR / "B3S_step1_per_image.csv")
assert len(td_df) == 400 and len(td_res) == 400
print("Repo commit:", REPO_COMMIT, "| output:", OUT_DIR)

Repo commit: d70bb1d6d631c12156ba7bf1881d22f5b2eb6100 | output: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3N_Negative_Point_Prompts


In [3]:
import torch

# ============================================================
# Step 1.2 — Shared helpers: guided masks, B0 meta filtering, systems, targets
# ============================================================

def guided_masks_for_prompts(predictor, image_np, prompts_xy):
    h, w = image_np.shape[:2]
    masks, scores = [], []
    if len(prompts_xy):
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p in prompts_xy:
                m, s, _ = predictor.predict(point_coords=np.asarray([p], dtype=np.float32),
                                            point_labels=np.asarray([1], dtype=np.int32),
                                            multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float)

def filter_meta(meta, keep):
    return {k: np.asarray(v)[keep] for k, v in meta.items()}

def systems_from_masks(b0_masks, b0_iou, g_masks, g_scores, a3c, depth, san_target):
    keep = gpv.nms_masks(g_masks, g_scores, 0.5) if g_masks.shape[0] else np.zeros(0, int)
    r4_m, _, _ = gpv.build_r4(g_masks[keep], np.asarray(g_scores)[keep], b0_masks, b0_iou, 0.5)
    d1 = gpv.d1_peaks(san_target)
    h1b = float(hybrid.h1b_count(r4_m, d1, a3c, depth, SG_CFG)[0])
    return {"B0": int(b0_masks.shape[0]), "GPV_R4": int(r4_m.shape[0]), "H1b": h1b}

def evaluate(frame, systems, label):
    gt = frame["gt_count"].to_numpy(float); st = frame["density_stratum"].to_numpy()
    AE = {s: np.abs(frame[s].to_numpy(float) - gt) for s in systems}
    IDX = {}
    for j, k in enumerate(STRATA):
        pos = np.flatnonzero(st == k)
        IDX[k] = pos[np.random.default_rng(1000 + j).integers(0, len(pos), size=(10_000, len(pos)))]
    def sci(a, b, k):
        d = AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)
        return AE[a][st == k].mean() - AE[b][st == k].mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)
    def rci(a, b):
        d = sum(w * (AE[a][IDX[k]].mean(1) - AE[b][IDX[k]].mean(1)) for k, w in W.items())
        o = sum(w * (AE[a][st == k].mean() - AE[b][st == k].mean()) for k, w in W.items())
        return o, np.percentile(d, 2.5), np.percentile(d, 97.5)
    res = pd.DataFrame([{"system": s,
                         "MAE_reweighted": sum(w * AE[s][st == k].mean() for k, w in W.items()),
                         "mean_signed": float((frame[s] - frame["gt_count"]).mean()),
                         **{f"MAE_{k[:2]}": AE[s][st == k].mean() for k in STRATA}} for s in systems])
    rows = []
    for s in [x for x in systems if x not in ("B0_v1", "GPV_R4_v1", "H1b_v1")]:
        rec = {"system": s}; t1 = []; t2 = []
        for k in ["S1_0_20", "S2_20_40"]:
            d, lo, hi = sci(s, "B0_v1", k); rec[f"{k[:2]} vs B0"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t1.append(lo <= 0)
        for k in ["S5_80_95", "S6_95_100"]:
            d, lo, hi = sci(s, "H1b_v1", k); rec[f"{k[:2]} vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"; t2.append(lo <= 0)
        d, lo, hi = rci(s, "H1b_v1"); rec["overall vs H1b"] = f"{d:+.2f} [{lo:+.2f}, {hi:+.2f}]"
        rec.update({"T1": all(t1), "T2'": all(t2), "T3'": hi < 0, "ALL": all(t1) and all(t2) and hi < 0})
        rows.append(rec)
    print(f"{label} — metrics:"); display(res.round(2))
    print("Targets (difference [95% CI]; negative = system better):"); display(pd.DataFrame(rows))
    return res, pd.DataFrame(rows)
print("helpers ready")

helpers ready


# 2. SAM2.1-Small

In [4]:
# ============================================================
# Step 2.1 — Install SAM2 (frozen commit)
# ============================================================

import torch
assert torch.cuda.is_available(), "Switch the runtime to T4."
DEVICE = "cuda"
%cd /content
if not Path("/content/sam2_repo/sam2/__init__.py").exists():
    !git clone -q https://github.com/facebookresearch/sam2.git /content/sam2_repo
    !git -C /content/sam2_repo checkout -q {b0.FROZEN_SAM2_COMMIT}
    %cd /content/sam2_repo
    !pip install -q -e .
    %cd /content
if "/content/sam2_repo" not in sys.path:
    sys.path.insert(0, "/content/sam2_repo")
importlib.invalidate_caches()
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor
print("SAM2 commit:", subprocess.check_output(["git", "-C", "/content/sam2_repo", "rev-parse", "HEAD"], text=True).strip())
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__, "| CUDA", torch.version.cuda)

/content
/content/sam2_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.8/168.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 9.4 MB/s eta 0:00:00
  Building editable for SAM-2 (pyproject.toml) ... done
/content
SAM2 commit: 2b90b9f5ceec907a1c18123530e92e794ad901a4
GPU: Tesla T4 | torch 2.11.0+cu130 | CUDA 13.0


In [5]:
# ============================================================
# Step 2.2 — Build the frozen SAM2.1-Small predictor
# ============================================================

SAM2_S = PROJECT_ROOT / "Checkpoints" / "SAM2" / "sam2.1_hiera_small.pt"
assert b0.sha256_file(SAM2_S) == b0.FROZEN_CHECKPOINT_SHA256
predictor = SAM2ImagePredictor(build_sam2(b0.FROZEN_MODEL_CFG, str(SAM2_S), device=DEVICE).eval())
print("SAM2.1-Small predictor ready")

SAM2.1-Small predictor ready


# 3. Negative-point guided masks

In [6]:
# ============================================================
# Step 3.1 — Guided masks with negative points (resumable)
# ============================================================

from scipy.spatial import cKDTree
MAX_NEG, RADIUS_FACTOR = 4, 2.0
NP_ROOT = TD_ROOT / "guided_negative_points"
for d in ["guided", "_done"]:
    (NP_ROOT / d).mkdir(parents=True, exist_ok=True)

def negative_sets(prompts):
    n = len(prompts)
    if n < 2:
        return [np.zeros((0, 2))] * n, np.nan
    tree = cKDTree(prompts)
    dist, idx = tree.query(prompts, k=min(n, MAX_NEG + 1))
    d_med = float(np.median(dist[:, 1]))
    out = []
    for i in range(n):
        sel = [j for d, j in zip(dist[i, 1:], idx[i, 1:]) if d <= RADIUS_FACTOR * d_med]
        out.append(prompts[sel] if sel else np.zeros((0, 2)))
    return out, d_med

def guided_masks_with_negatives(image_np, prompts):
    h, w = image_np.shape[:2]
    masks, scores, n_negs = [], [], []
    if len(prompts):
        negs, _ = negative_sets(prompts)
        with torch.inference_mode():
            predictor.set_image(image_np)
            for p, ng in zip(prompts, negs):
                coords = np.vstack([p[None], ng]).astype(np.float32)
                labels = np.array([1] + [0] * len(ng), dtype=np.int32)
                m, s, _ = predictor.predict(point_coords=coords, point_labels=labels, multimask_output=False)
                masks.append(np.asarray(m[0], dtype=bool)); scores.append(float(np.asarray(s)[0]))
                n_negs.append(len(ng))
    return (np.asarray(masks) if masks else np.zeros((0, h, w), bool)), np.asarray(scores, float), n_negs

todo = [r for r in td_df.itertuples(index=False)
        if not (NP_ROOT / "_done" / f"{int(r.dataset_index):05d}.json").exists()]
print(f"Done: {len(td_df) - len(todo)} | to process: {len(todo)}")
t0 = time.time()
for i, r in enumerate(tqdm(todo, desc="Negative-point prompts")):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    img = b0.load_image_rgb_cv2(FSC147_IMAGE_DIR / r.image_id)
    prompts = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")["prompts_xy"]
    gm, gs, nn = guided_masks_with_negatives(img, prompts)
    io_cache.save_guided_masks(NP_ROOT / "guided" / f"{int(di):04d}_{stem}.npz", gm, prompts, gs,
                               img.shape[:2], dataset_index=di, image_id=r.image_id)
    with open(NP_ROOT / "_done" / f"{int(di):05d}.json", "w") as f:
        json.dump({"dataset_index": int(di), "n_prompts": int(len(prompts)),
                   "mean_negatives": float(np.mean(nn)) if nn else 0.0}, f)
    if (i + 1) % 50 == 0:
        torch.cuda.empty_cache()
print(f"Elapsed: {(time.time() - t0) / 60:.1f} min")

Done: 0 | to process: 400


Negative-point prompts:   0%|          | 0/400 [00:00<?, ?it/s]

Elapsed: 9.6 min


# 4. Systems and targets

In [7]:
# ============================================================
# Step 4.1 — GPV_R4 / H1b with negative-point guided masks; targets
# ============================================================

NP_ROOT = TD_ROOT / "guided_negative_points"
rows = []
for r in tqdm(td_df.itertuples(index=False), total=len(td_df), desc="Systems"):
    di, stem = r.dataset_index, io_cache.image_stem(r.image_id)
    p = io_cache.cache_paths(di, r.image_id, TD_DIRS)
    b0m, b0meta = io_cache.load_b0_masks(p["b0_masks"])                  # frozen B0 (unchanged)
    a3c, _ = io_cache.load_a3c_mask(p["a3c_masks"])
    dep = io_cache.load_depth(p["depth"])
    san = io_cache.load_san_scores(TD_ROOT / "san" / io_cache.san_scores_filename(r.image_id))
    g = io_cache.load_guided_masks(NP_ROOT / "guided" / f"{int(di):04d}_{stem}.npz")
    g0 = io_cache.load_guided_masks(TD_ROOT / "guided_dense_recovery" / f"{int(di):04d}_{stem}.npz")
    s = systems_from_masks(b0m, b0meta["predicted_iou"], g["masks"], g["scores"], a3c, dep, san["target_score"])
    info = json.load(open(NP_ROOT / "_done" / f"{int(di):05d}.json"))
    area_new = float(np.median(g["masks"].reshape(len(g["masks"]), -1).sum(1))) if len(g["masks"]) else np.nan
    area_old = float(np.median(g0["masks"].reshape(len(g0["masks"]), -1).sum(1))) if len(g0["masks"]) else np.nan
    rows.append({"dataset_index": di, "GPV_R4[neg]": s["GPV_R4"], "H1b[neg]": s["H1b"],
                 "mean_negatives": info["mean_negatives"],
                 "median_guided_area_old": area_old, "median_guided_area_neg": area_new})
nx = pd.DataFrame(rows).merge(
    td_res[["dataset_index", "gt_count", "density_stratum", "B0", "GPV_R4_v1", "H1b_v1"]]
    .rename(columns={"B0": "B0_v1"}), on="dataset_index", validate="one_to_one")

print("Negatives per prompt and median guided-mask area (old vs negative-point), by stratum:")
display(nx.groupby("density_stratum")[["mean_negatives", "median_guided_area_old", "median_guided_area_neg"]]
        .mean().reindex(STRATA).round(1))

SYS = ["B0_v1", "GPV_R4_v1", "H1b_v1", "GPV_R4[neg]", "H1b[neg]"]
res, targets = evaluate(nx, SYS, "Test B — negative-point prompting (train_dev400)")

nx.to_csv(OUT_DIR / "B3N_per_image.csv", index=False)
res.to_csv(OUT_DIR / "B3N_metrics.csv", index=False)
targets.to_csv(OUT_DIR / "B3N_targets.csv", index=False)
with open(OUT_DIR / "B3N_metadata.json", "w") as f:
    json.dump({"notebook": "B3N_Negative_Point_Prompts", "test": "B — negative-point prompting",
               "negatives": f"up to {MAX_NEG} nearest other prompts within {RADIUS_FACTOR} x median NN prompt distance"
               if "MAX_NEG" in globals() else "up to 4 nearest within 2 x median NN distance",
               "model": "SAM2.1-Small, full image, multimask off", "b0": "frozen cache",
               "counting": "GPV_R4 (NMS 0.5 + B0 pool), H1b on those masks",
               "date_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
               "repo_commit": REPO_COMMIT, "heldout_accessed": False, "test_set_accessed": False}, f, indent=2)
print("Saved to:", OUT_DIR)

Systems:   0%|          | 0/400 [00:00<?, ?it/s]

Negatives per prompt and median guided-mask area (old vs negative-point), by stratum:


,mean_negatives,median_guided_area_old,median_guided_area_neg
density_stratum,,,
S1_0_20,2.6,17457.1,5492.3
S2_20_40,2.8,19430.1,4870.9
S3_40_60,2.9,12895.9,4114.8
S4_60_80,3.1,6012.6,2831.5
S5_80_95,3.0,5942.2,2031.9
S6_95_100,3.1,26609.1,1871.4


Test B — negative-point prompting (train_dev400) — metrics:


,system,MAE_reweighted,mean_signed,MAE_S1,MAE_S2,MAE_S3,MAE_S4,MAE_S5,MAE_S6
0,B0_v1,35.67,-62.67,5.56,7.74,13.30,22.93,59.03,338.08
1,GPV_R4_v1,29.94,-46.78,8.62,7.70,12.45,16.13,40.52,297.63
2,H1b_v1,28.65,-41.28,10.46,9.49,12.35,14.20,37.33,274.98
3,GPV_R4[neg],38.08,-32.52,19.06,19.18,27.98,24.78,40.17,277.00
4,H1b[neg],36.95,-30.90,19.19,19.56,27.60,22.87,38.42,266.87


Targets (difference [95% CI]; negative = system better):


,system,S1 vs B0,S2 vs B0,S5 vs H1b,S6 vs H1b,overall vs H1b,T1,T2',T3',ALL
0,GPV_R4[neg],"+13.50 [+10.39, +16.81]","+11.44 [+8.25, +14.79]","+2.83 [-2.23, +8.92]","+2.02 [-4.95, +9.82]","+9.43 [+7.61, +11.26]",False,True,False,False
1,H1b[neg],"+13.62 [+10.64, +16.70]","+11.82 [+8.64, +15.16]","+1.08 [-3.17, +5.62]","-8.12 [-12.67, -3.15]","+8.30 [+6.76, +9.91]",False,True,False,False


Saved to: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B3N_Negative_Point_Prompts
